# Running the Experiment

This experiment is implemented as a single end-to-end training and evaluation pipeline. The goal is to minimize manual intervention while ensuring reproducibility. The script integrates dataset loading, preprocessing, augmentation, model training, validation, threshold tuning, and final evaluation.

---

## Pipeline Overview

The execution pipeline consists of the following steps:

1. **Environment Setup**
   - Automatically selects computation device (GPU if available, otherwise CPU)
   - Sets deterministic random seeds for Python, NumPy, and PyTorch

2. **Dataset Loading**
   - Downloads the dataset `DaniilOr/SemEval-2026-Task13` (configuration `A`)
   - Falls back to default configuration if needed
   - Automatically detects relevant columns:
     - code
     - label
     - language (optional)

3. **Data Preprocessing**
   - Normalizes labels into binary format:
     - `0` → human-written
     - `1` → machine-generated
   - Cleans code formatting:
     - removes trailing spaces
     - normalizes line endings
     - reduces excessive blank lines

4. **Data Augmentation (Training Only)**
   - Applies stochastic transformations:
     - identifier renaming
     - dead-comment insertion
     - dead-code insertion
     - whitespace perturbation
   - Improves robustness to superficial variations

5. **Input Representation**
   - Each sample is transformed into:
     - a language token (e.g., `<LANG_PY>`)
     - metadata tokens (length and comment presence)
     - the cleaned source code
   - Tokens are prepended to the input sequence

6. **Tokenization**
   - Uses `microsoft/codebert-base` tokenizer
   - Extends vocabulary with custom special tokens
   - Applies truncation at maximum length of 512

---

## Model Architecture

The model is based on **CodeBERT** with additional components:

- Encoder: `microsoft/codebert-base`
- Classification head:
  - MLP with GELU activation and dropout
- Adversarial language head:
  - Predicts programming language
  - Uses gradient reversal to enforce language-invariant features

---

## Training Configuration

- Max sequence length: `512`
- Batch size: `16`
- Gradient accumulation: `2`
- Effective batch size: `32`
- Epochs: `2`
- Learning rate: `2e-5`
- Weight decay: `0.01`
- Warmup ratio: `0.06`
- Scheduler: cosine
- Dropout: `0.2`
- Mixed precision: enabled if GPU is available
- Adversarial loss weight: `0.15`

---

## Training Details

- Binary classification using sigmoid + BCE loss
- Additional adversarial loss for language prediction
- Combined loss = classification loss + adversarial loss
- Dynamic padding via Hugging Face data collator
- Evaluation performed periodically during training

---

## Validation and Threshold Tuning

After training:

- Predictions are converted to probabilities using sigmoid
- Default threshold: `0.5`
- Additional threshold search is performed:
  - scans values in `[0, 1]`
  - selects threshold maximizing macro F1-score
- Best threshold is saved to `best_threshold.json`

---

## Test Evaluation

The model is evaluated on the official test split:

- Applies same preprocessing pipeline (without augmentation)
- Uses saved optimal threshold (or 0.5 fallback)
- Computes:
  - macro F1-score
  - F1 for human class (0)
  - F1 for machine class (1)
  - confusion matrix
  - full classification report

Additionally:

- An optional best threshold is computed directly on test data
- This is used only for analysis, not for official reporting

---

## Output

At the end of the execution, the script prints the final evaluation results on the test set, including:

- macro F1-score  
- per-class F1-scores  
- confusion matrix  
- classification report  
- best threshold on test (optional)

**The expected output of the experiment is displayed at the end of the cell.**

In [ ]:

!pip -q install -U "transformers>=4.41.0" "datasets>=2.19.0" "accelerate>=0.31.0" "evaluate>=0.4.2" "scikit-learn>=1.3.0" "tokenizers>=0.19.1"

import os, re, math, random, json
from dataclasses import dataclass
from typing import Dict, Any, Optional, List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoConfig,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed,
)
from sklearn.metrics import f1_score
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
DATASET_NAME = "DaniilOr/SemEval-2026-Task13"
CONFIG = "A"

try:
    ds = load_dataset(DATASET_NAME, CONFIG)
except Exception as e:
    print("Could not load config 'A'. Falling back to default config. Error:", e)
    ds = load_dataset(DATASET_NAME)

print(ds)
def detect_columns(split_ds) -> Dict[str, Optional[str]]:
    cols = split_ds.column_names
    lower = {c.lower(): c for c in cols}

    def pick(*candidates):
        for cand in candidates:
            if cand in cols:
                return cand
            if cand.lower() in lower:
                return lower[cand.lower()]
        return None

    code_col  = pick("code", "content", "snippet", "text", "source", "src")
    label_col = pick("label", "labels", "target", "y", "is_machine", "machine", "generated")
    lang_col  = pick("language", "lang", "pl", "prog_lang")
    dom_col   = pick("domain", "split_domain", "category", "source_domain")

    return {"code": code_col, "label": label_col, "lang": lang_col, "domain": dom_col}

colmap = detect_columns(ds["train"])
print("Detected columns:", colmap)

assert colmap["code"] is not None, "Could not find code column. Please check dataset columns."
assert colmap["label"] is not None, "Could not find label column. Please check dataset columns."

CODE_COL, LABEL_COL = colmap["code"], colmap["label"]
LANG_COL, DOMAIN_COL = colmap["lang"], colmap["domain"]

def normalize_label(x):
    if isinstance(x, (int, np.integer)):
        return int(x)
    if isinstance(x, (float, np.floating)):
        return int(x)
    if isinstance(x, str):
        s = x.strip().lower()
        if s in ["0", "human", "written", "human_written", "human-written", "hum", "false"]:
            return 0
        if s in ["1", "machine", "generated", "ai", "llm", "true", "machine_generated", "machine-generated"]:
            return 1
        try:
            return int(s)
        except:
            pass
    return int(x)

def basic_clean(code: str) -> str:
    if code is None:
        return ""
    code = code.replace("\r\n", "\n").replace("\r", "\n")
    code = re.sub(r"[ \t]+$", "", code, flags=re.MULTILINE)
    code = re.sub(r"\n{4,}", "\n\n\n", code)
    return code.strip()
IDENT_RE = re.compile(r"\b([A-Za-z_][A-Za-z_0-9]{2,})\b")

PY_KEYWORDS = set("""
False None True and as assert async await break class continue def del elif else except finally for from global if import in is lambda
nonlocal not or pass raise return try while with yield
""".split())

JAVA_KEYWORDS = set("""
abstract assert boolean break byte case catch char class const continue default do double else enum extends final finally float for goto if implements import
instanceof int interface long native new package private protected public return short static strictfp super switch synchronized this throw throws transient try void volatile while
""".split())

CPP_KEYWORDS = set("""
alignas alignof and and_eq asm atomic_cancel atomic_commit atomic_noexcept auto bitand bitor bool break case catch char char16_t char32_t class compl concept const
consteval constexpr constinit const_cast continue co_await co_return co_yield decltype default delete do double dynamic_cast else enum explicit export extern false float for friend goto
if inline int long mutable namespace new noexcept not not_eq nullptr operator or or_eq private protected public register reinterpret_cast requires return short signed sizeof static static_assert
static_cast struct switch template this thread_local throw true try typedef typeid typename union unsigned using virtual void volatile wchar_t while xor xor_eq
""".split())

def rename_identifiers(code: str, lang: Optional[str]) -> str:
    if not code:
        return code
    lang_l = (lang or "").lower()
    if "python" in lang_l or lang_l == "py":
        keywords = PY_KEYWORDS
    elif "java" in lang_l:
        keywords = JAVA_KEYWORDS
    else:
        keywords = CPP_KEYWORDS

    words = list(dict.fromkeys(IDENT_RE.findall(code)))
    cands = [w for w in words if w not in keywords and len(w) >= 3]
    if len(cands) < 3:
        return code

    k = min(6, max(2, len(cands)//6))
    random.shuffle(cands)
    chosen = cands[:k]

    mapping = {}
    for i, w in enumerate(chosen):
        mapping[w] = f"v{i+1}"

    def repl(m):
        w = m.group(1)
        return mapping.get(w, w)

    return IDENT_RE.sub(repl, code)

def add_dead_comment(code: str, lang: Optional[str]) -> str:
    if not code:
        return code
    lang_l = (lang or "").lower()
    if "python" in lang_l:
        comment = "# note: helper placeholder\n"
    else:
        comment = "// note: helper placeholder\n"
    if code.startswith("#!") and "\n" in code:
        first, rest = code.split("\n", 1)
        return first + "\n" + comment + rest
    return comment + code

def add_dead_code_stub(code: str, lang: Optional[str]) -> str:
    if not code:
        return code
    lang_l = (lang or "").lower()
    if "python" in lang_l:
        stub = "\n\ndef __unused_helper(x):\n    return x\n"
    elif "java" in lang_l:
        stub = "\n\n// unused helper\nstatic int __unused_helper(int x){ return x; }\n"
    else:
        stub = "\n\n// unused helper\nstatic int __unused_helper(int x){ return x; }\n"
    return code + stub

def random_whitespace_perturb(code: str) -> str:
    if not code:
        return code
    lines = code.split("\n")
    out = []
    for ln in lines:
        if random.random() < 0.08:
            ln = re.sub(r"\s{2,}", " ", ln)
        out.append(ln)
        if random.random() < 0.03:
            out.append("")
    return "\n".join(out)

def augment_code(code: str, lang: Optional[str]) -> str:
    code = basic_clean(code)
    if random.random() < 0.20:
        code = rename_identifiers(code, lang)
    if random.random() < 0.10:
        code = add_dead_comment(code, lang)
    if random.random() < 0.08:
        code = add_dead_code_stub(code, lang)
    if random.random() < 0.20:
        code = random_whitespace_perturb(code)
    return basic_clean(code)
USE_LANG_TOKEN = True
USE_META_TOKENS = True

LANG_TOKENS = {
    "python": "<LANG_PY>",
    "py": "<LANG_PY>",
    "java": "<LANG_JAVA>",
    "c++": "<LANG_CPP>",
    "cpp": "<LANG_CPP>",
    "c": "<LANG_C>",
    "c#": "<LANG_CS>",
    "cs": "<LANG_CS>",
    "go": "<LANG_GO>",
    "golang": "<LANG_GO>",
    "php": "<LANG_PHP>",
    "javascript": "<LANG_JS>",
    "js": "<LANG_JS>",
}

def lang_to_token(lang: Optional[str]) -> str:
    if not lang:
        return "<LANG_UNK>"
    s = str(lang).strip().lower()
    return LANG_TOKENS.get(s, "<LANG_UNK>")

def meta_tokens(code: str, lang: Optional[str]) -> List[str]:
    toks = []
    n = len(code)
    if n < 400:
        toks.append("<LEN_S>")
    elif n < 1200:
        toks.append("<LEN_M>")
    else:
        toks.append("<LEN_L>")

    l = (lang or "").lower()
    has_comment = ("#" in code) if "python" in l else ("//" in code or "/*" in code)
    toks.append("<HAS_CMT>" if has_comment else "<NO_CMT>")
    return toks

class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambd * grad_output, None

def grad_reverse(x, lambd: float):
    return GradReverse.apply(x, lambd)

class CodeBERTBinaryAdv(nn.Module):
    def __init__(self, base_model_name: str, num_langs: int = 8, adv: bool = True, adv_lambda: float = 0.2):
        super().__init__()
        self.adv = adv
        self.adv_lambda = adv_lambda

        self.encoder = AutoModel.from_pretrained(base_model_name)
        hidden = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(0.2)
        self.cls_head = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, 1)
        )

        if self.adv:
            self.lang_head = nn.Sequential(
                nn.Linear(hidden, hidden//2),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(hidden//2, num_langs)
            )

    def forward(self, input_ids, attention_mask, labels=None, lang_ids=None):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = out.last_hidden_state[:, 0, :]
        pooled = self.dropout(pooled)

        logits = self.cls_head(pooled).squeeze(-1)  # (B,)
        loss = None
        lang_logits = None

        if labels is not None:
            loss = F.binary_cross_entropy_with_logits(logits, labels.float())

        if self.adv and (lang_ids is not None):
            rev = grad_reverse(pooled, self.adv_lambda)
            lang_logits = self.lang_head(rev)
            lang_loss = F.cross_entropy(lang_logits, lang_ids.long())
            loss = loss + lang_loss if loss is not None else lang_loss

        return {"loss": loss, "logits": logits, "lang_logits": lang_logits}
BASE_MODEL = "microsoft/codebert-base"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)

specials = ["<LANG_PY>", "<LANG_JAVA>", "<LANG_CPP>", "<LANG_C>", "<LANG_CS>", "<LANG_GO>", "<LANG_PHP>", "<LANG_JS>", "<LANG_UNK>",
            "<LEN_S>", "<LEN_M>", "<LEN_L>", "<HAS_CMT>", "<NO_CMT>"]
tokenizer.add_special_tokens({"additional_special_tokens": specials})

LANG_ID_TABLE = {
    "<LANG_UNK>": 0,
    "<LANG_PY>": 1,
    "<LANG_JAVA>": 2,
    "<LANG_CPP>": 3,
    "<LANG_C>": 4,
    "<LANG_CS>": 5,
    "<LANG_GO>": 6,
    "<LANG_PHP>": 7,
    "<LANG_JS>": 8,
}
NUM_LANGS = max(LANG_ID_TABLE.values()) + 1
MAX_LEN = 512
AUGMENT_TRAIN = True

def build_input(code: str, lang: Optional[str]) -> Tuple[str, int]:
    code = basic_clean(code)
    lang_tok = lang_to_token(lang)
    lang_id = LANG_ID_TABLE.get(lang_tok, 0)

    prefix = []
    if USE_LANG_TOKEN:
        prefix.append(lang_tok)
    if USE_META_TOKENS:
        prefix.extend(meta_tokens(code, lang))

    if prefix:
        text = " ".join(prefix) + "\n" + code
    else:
        text = code
    return text, lang_id

def preprocess_batch(examples, split_name="train"):
    codes = examples[CODE_COL]
    labels = examples[LABEL_COL]
    langs = examples[LANG_COL] if LANG_COL and (LANG_COL in examples) else [None]*len(codes)

    out_texts = []
    out_labels = []
    out_lang_ids = []

    for code, y, lang in zip(codes, labels, langs):
        y = normalize_label(y)

        if split_name == "train" and AUGMENT_TRAIN:
            code2 = augment_code(code, lang)
        else:
            code2 = basic_clean(code)

        text, lang_id = build_input(code2, lang)
        out_texts.append(text)
        out_labels.append(y)
        out_lang_ids.append(lang_id)

    tok = tokenizer(out_texts, truncation=True, max_length=MAX_LEN)
    tok["labels"] = out_labels
    tok["lang_ids"] = out_lang_ids
    return tok

splits = list(ds.keys())
print("Splits:", splits)

if "validation" not in ds and "val" in ds:
    ds = DatasetDict(train=ds["train"], validation=ds["val"])
elif "validation" not in ds:
    ds = ds["train"].train_test_split(test_size=0.17, seed=SEED)
    ds = DatasetDict(train=ds["train"], validation=ds["test"])

train_ds = ds["train"].map(lambda e: preprocess_batch(e, "train"), batched=True, remove_columns=ds["train"].column_names)
val_ds   = ds["validation"].map(lambda e: preprocess_batch(e, "validation"), batched=True, remove_columns=ds["validation"].column_names)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)
def unwrap_logits(predictions):
    """
    HF Trainer can return predictions as:
      - np.ndarray logits
      - tuple(list) of arrays (e.g., (cls_logits, lang_logits))
    We need ONLY cls logits for binary detection.
    """
    if isinstance(predictions, (tuple, list)):
        predictions = predictions[0]
    arr = np.asarray(predictions)
    return arr.squeeze()

def compute_metrics(eval_pred):
    logits = unwrap_logits(eval_pred.predictions)
    labels = eval_pred.label_ids
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    macro = f1_score(labels, preds, average="macro")
    f1_0 = f1_score(labels, preds, pos_label=0, average="binary")
    f1_1 = f1_score(labels, preds, pos_label=1, average="binary")
    return {"macro_f1@0.5": macro, "f1_human": f1_0, "f1_machine": f1_1}

def find_best_threshold(logits: np.ndarray, labels: np.ndarray, steps: int = 101) -> Tuple[float, float]:
    logits = unwrap_logits(logits)
    probs = 1 / (1 + np.exp(-logits))
    best_t, best_f1 = 0.5, -1.0
    for t in np.linspace(0.0, 1.0, steps):
        preds = (probs >= t).astype(int)
        mf1 = f1_score(labels, preds, average="macro")
        if mf1 > best_f1:
            best_f1 = mf1
            best_t = float(t)
    return best_t, float(best_f1)

class AdvTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        lang_ids = inputs.get("lang_ids")
        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            labels=labels,
            lang_ids=lang_ids
        )
        loss = outputs["loss"]
        return (loss, outputs) if return_outputs else loss

USE_ADVERSARIAL = True
ADV_LAMBDA = 0.15

model = CodeBERTBinaryAdv(BASE_MODEL, num_langs=NUM_LANGS, adv=USE_ADVERSARIAL, adv_lambda=ADV_LAMBDA)
model.encoder.resize_token_embeddings(len(tokenizer))
model.to(device)

BATCH = 16
GRAD_ACC = 2
EPOCHS = 2
LR = 2e-5
WDECAY = 0.01
WARMUP = 0.06

args = TrainingArguments(
    output_dir="./codebert_taskA",
    learning_rate=LR,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACC,
    num_train_epochs=EPOCHS,
    weight_decay=WDECAY,
    warmup_ratio=WARMUP,
    lr_scheduler_type="cosine",
    eval_strategy="steps",
    eval_steps=2000,
    save_steps=2000,
    logging_steps=200,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
    metric_for_best_model="macro_f1@0.5",
    load_best_model_at_end=True,
)

trainer = AdvTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
train_out = trainer.train()
print(train_out)

pred = trainer.predict(val_ds)
val_logits = unwrap_logits(pred.predictions)
val_labels = pred.label_ids
best_t, best_f1 = find_best_threshold(val_logits, val_labels, steps=201)
print("Best threshold:", best_t, "Best macro F1:", best_f1)

with open(os.path.join(args.output_dir, "best_threshold.json"), "w") as f:
    json.dump({"threshold": best_t, "macro_f1": best_f1}, f)

def predict_split(split_ds, threshold: float):
    pred = trainer.predict(split_ds)
    logits = unwrap_logits(pred.predictions)
    probs = 1 / (1 + np.exp(-logits))
    yhat = (probs >= threshold).astype(int)
    return yhat, probs

"""
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

def get_raw(split, max_n=None):
    raw = ds[split]
    if max_n:
        raw = raw.select(range(min(max_n, len(raw))))
    X = [basic_clean(x) for x in raw[CODE_COL]]
    y = [normalize_label(x) for x in raw[LABEL_COL]]
    return X, y

Xtr, ytr = get_raw("train", max_n=80000)
Xva, yva = get_raw("validation", max_n=20000)

vec = TfidfVectorizer(analyzer="char", ngram_range=(3,6), min_df=2, max_features=400000)
Xtrv = vec.fit_transform(Xtr)
Xvav = vec.transform(Xva)

clf = LinearSVC()
clf.fit(Xtrv, ytr)
pred = clf.predict(Xvav)
print("TFIDF+LinearSVC macro F1:", f1_score(yva, pred, average="macro"))
"""


In [ ]:

import os, json
import numpy as np
from datasets import load_dataset, DatasetDict
from sklearn.metrics import f1_score, classification_report, confusion_matrix

DATASET_NAME = "DaniilOr/SemEval-2026-Task13"
CONFIG = "A"
dsA = load_dataset(DATASET_NAME, CONFIG)
assert "test" in dsA, f"No 'test' split found. Available: {list(dsA.keys())}"
raw_test = dsA["test"]
print("Raw test columns:", raw_test.column_names)
print("Test size:", len(raw_test))
def detect_columns(split_ds):
    cols = split_ds.column_names
    lower = {c.lower(): c for c in cols}
    def pick(*candidates):
        for cand in candidates:
            if cand in cols:
                return cand
            if cand.lower() in lower:
                return lower[cand.lower()]
        return None

    code_col  = pick("code", "content", "snippet", "text", "source", "src")
    label_col = pick("label", "labels", "target", "y", "is_machine", "machine", "generated")
    lang_col  = pick("language", "lang", "pl", "prog_lang")
    id_col    = pick("id", "idx", "sample_id", "example_id", "uid")
    return code_col, label_col, lang_col, id_col

CODE_COL_T, LABEL_COL_T, LANG_COL_T, ID_COL_T = detect_columns(raw_test)
print("Detected:", {"code": CODE_COL_T, "label": LABEL_COL_T, "lang": LANG_COL_T, "id": ID_COL_T})

assert CODE_COL_T is not None, "Nu găsesc coloana cu codul."
assert LABEL_COL_T is not None, "Nu găsesc coloana de label în test. Dacă chiar există, spune-mi numele exact."

def normalize_label(x):
    if isinstance(x, (int, np.integer)): return int(x)
    if isinstance(x, (float, np.floating)): return int(x)
    if isinstance(x, str):
        s = x.strip().lower()
        if s in ["0", "human", "written", "human_written", "human-written", "hum", "false"]:
            return 0
        if s in ["1", "machine", "generated", "ai", "llm", "true", "machine_generated", "machine-generated"]:
            return 1
        try: return int(s)
        except: pass
    return int(x)
threshold = 0.5
if "best_t" in globals():
    threshold = float(best_t)
else:
    th_path = os.path.join(trainer.args.output_dir, "best_threshold.json")
    if os.path.exists(th_path):
        with open(th_path, "r") as f:
            threshold = float(json.load(f)["threshold"])
print("Using threshold:", threshold)
MAX_LEN = 512

def preprocess_test_batch(examples):
    codes = examples[CODE_COL_T]
    labels = [normalize_label(x) for x in examples[LABEL_COL_T]]
    langs = examples[LANG_COL_T] if (LANG_COL_T and LANG_COL_T in examples) else [None]*len(codes)

    texts = []
    lang_ids = []
    for code, lang in zip(codes, langs):
        code2 = basic_clean(code)
        text, lang_id = build_input(code2, lang)
        texts.append(text)
        lang_ids.append(lang_id)

    tok = tokenizer(texts, truncation=True, max_length=MAX_LEN)
    tok["labels"] = labels
    tok["lang_ids"] = lang_ids
    return tok

test_ds = raw_test.map(
    preprocess_test_batch,
    batched=True,
    remove_columns=raw_test.column_names
)

pred = trainer.predict(test_ds)
logits = unwrap_logits(pred.predictions)
labels = pred.label_ids

probs = 1 / (1 + np.exp(-logits))
yhat = (probs >= threshold).astype(int)

macro = f1_score(labels, yhat, average="macro")
f1_0  = f1_score(labels, yhat, pos_label=0, average="binary")
f1_1  = f1_score(labels, yhat, pos_label=1, average="binary")

print("\n=== TEST METRICS (fixed threshold) ===")
print("macro_f1:", macro)
print("f1_human(0):", f1_0)
print("f1_machine(1):", f1_1)
print("\nConfusion matrix [ [TN FP], [FN TP] ]:\n", confusion_matrix(labels, yhat))
print("\nClassification report:\n", classification_report(labels, yhat, digits=4))

def best_threshold_on_split(logits, labels, steps=201):
    probs = 1 / (1 + np.exp(-logits))
    best_t, best_f1 = 0.5, -1.0
    for t in np.linspace(0.0, 1.0, steps):
        p = (probs >= t).astype(int)
        mf1 = f1_score(labels, p, average="macro")
        if mf1 > best_f1:
            best_f1 = mf1
            best_t = float(t)
    return best_t, float(best_f1)

bt, bf1 = best_threshold_on_split(logits, labels, steps=401)
print("\n=== (Optional) BEST threshold on TEST ===")
print("best_t_test:", bt, "best_macro_f1_test:", bf1)


Raw test columns: ['code', 'generator', 'label', 'language']
Test size: 1000
Detected: {'code': 'code', 'label': 'label', 'lang': 'language', 'id': None}
Using threshold: 0.25


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]


=== TEST METRICS (fixed threshold) ===
macro_f1: 0.2969972613458529
f1_human(0): 0.2013888888888889
f1_machine(1): 0.3926056338028169

Confusion matrix [ [TN FP], [FN TP] ]:
 [[ 87 690]
 [  0 223]]

Classification report:
               precision    recall  f1-score   support

           0     1.0000    0.1120    0.2014       777
           1     0.2442    1.0000    0.3926       223

    accuracy                         0.3100      1000
   macro avg     0.6221    0.5560    0.2970      1000
weighted avg     0.8315    0.3100    0.2440      1000


=== (Optional) BEST threshold on TEST ===
best_t_test: 1.0 best_macro_f1_test: 0.43725379853685986
